In [ ]:
import os
import sys
import json
import csv
import time
import heapq
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from scipy.ndimage import binary_dilation
from collections import deque
from tqdm import tqdm
import matplotlib.pyplot as plt
import platform
import warnings

warnings.filterwarnings('ignore')

try:
    torch.set_num_threads(1)
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass  

# 1. КОНФИГУРАЦИЯ ЭКСПЕРИМЕНТА
CONFIG = {
    "grid_size": 30,
    "patch_size": 30,
    "densities": [0.10, 0.20, 0.30, 0.40, 0.50],
    "maps_per_density": 500,  
    "seed": 42,
    "device": "cpu",
    "cnn_epochs": 10,
    "warmup_runs": 3,
    "measurement_runs": 3,   
}

def get_env_info():
    return {
        "os": f"{platform.system()} {platform.release()}",
        "processor": platform.processor(),
        "python_version": platform.python_version(),
        "torch_version": torch.__version__,
        "note": "Single-threaded CPU mode is a REPRODUCIBILITY CONDITION."
    }

# 2. ГЕНЕРАЦИЯ СРЕДЫ И ПРАВИЛА ДВИЖЕНИЯ
def generate_map(size, density, rng):
    noise = rng.random((size, size)) < (density * 0.5)
    seeds = rng.random((size, size)) < (density * 0.1)
    clusters = binary_dilation(seeds, iterations=3)
    grid = (noise | clusters).astype(np.uint8)
    grid[0, 0] = 0
    grid[size-1, size-1] = 0
    return grid

def get_neighbors_8way_strict(grid, current):
    """Строгая 8-связность: запрет диагонального перехода, если занят ортогональный сосед"""
    x, y = current
    size = grid.shape[0]
    neighbors = []
    for dx in [-1, 0, 1]:
        for dy in [-1, 0, 1]:
            if dx == 0 and dy == 0: continue
            nx, ny = x + dx, y + dy
            if 0 <= nx < size and 0 <= ny < size and grid[nx, ny] == 0:
                if dx != 0 and dy != 0:
                    if grid[x+dx, y] == 1 or grid[x, y+dy] == 1: continue
                cost = 1.41421356 if (dx != 0 and dy != 0) else 1.0
                neighbors.append(((nx, ny), cost))
    return neighbors

def is_reachable_strict(grid, start, goal):
    size = grid.shape[0]
    visited = np.zeros((size, size), dtype=bool)
    queue = deque([start])
    visited[start] = True
    while queue:
        x, y = queue.popleft()
        if (x, y) == goal: return True
        for neighbor, _ in get_neighbors_8way_strict(grid, (x, y)):
            if not visited[neighbor]:
                visited[neighbor] = True
                queue.append(neighbor)
    return False

def compute_dijkstra_heuristic_map(grid, goal):
    """Вычисляет точную стоимость пути (h*) от ВСЕХ узлов до цели (обратный Дейкстра)"""
    size = grid.shape[0]
    h_star = np.full((size, size), np.inf)
    h_star[goal] = 0
    pq = [(0, goal)]
    
    while pq:
        cost, (x, y) = heapq.heappop(pq)
        if cost > h_star[x, y]:
            continue
        for dx in [-1, 0, 1]:
            for dy in [-1, 0, 1]:
                if dx == 0 and dy == 0: continue
                nx, ny = x + dx, y + dy
                if 0 <= nx < size and 0 <= ny < size and grid[nx, ny] == 0:
                    if dx != 0 and dy != 0:
                        if grid[x+dx, y] == 1 or grid[x, y+dy] == 1: continue
                    move_cost = 1.41421356 if (dx != 0 and dy != 0) else 1.0
                    new_cost = cost + move_cost
                    if new_cost < h_star[nx, ny]:
                        h_star[nx, ny] = new_cost
                        heapq.heappush(pq, (new_cost, (nx, ny)))
    return h_star

# 3. CNN МОДЕЛЬ И DATASET
class HeuristicDataset(Dataset):
    def __init__(self, maps, h_stars, patch_size):
        self.samples = []
        self.patch_size = patch_size
        pad = patch_size // 2
        
        for grid, h_star in zip(maps, h_stars):
            size = grid.shape[0]
            goal = (size-1, size-1)
            y_coords, x_coords = np.meshgrid(np.arange(size), np.arange(size), indexing='ij')
            
            max_dist = np.sqrt(2 * (patch_size**2)) 
            dist_to_goal = np.sqrt((x_coords - goal[1])**2 + (y_coords - goal[0])**2)
            dist_to_goal = np.clip(dist_to_goal / max_dist, 0.0, 1.0)
            
            padded_grid = np.pad(grid, pad, mode='constant', constant_values=1)
            padded_dist = np.pad(dist_to_goal, pad, mode='constant', constant_values=1.0)
            
            for x in range(size):
                for y in range(size):
                    if grid[x, y] == 0 and h_star[x, y] != np.inf:
                        patch_obs = padded_grid[x : x+patch_size, y : y+patch_size]
                        patch_dist = padded_dist[x : x+patch_size, y : y+patch_size]
                        sample = np.stack([patch_obs, patch_dist], axis=0)
                        self.samples.append((sample, h_star[x, y]))

    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        x, y = self.samples[idx]
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

class LightweightCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(2, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 16, kernel_size=3, padding=1)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(16, 1) 
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.relu(self.conv3(x))
        x = self.pool(x)
        return self.fc(torch.flatten(x, 1))

# 4. АЛГОРИТМЫ С ДЕТАЛЬНЫМ ПРОФИЛИРОВАНИЕМ
def astar_profiled(grid, start, goal, cnn_model=None, patch_size=30):
    size = grid.shape[0]
    open_set = [(0, 0, 0, start)] # f, g, counter, node
    g_score = {start: 0}
    came_from = {}
    counter = 1
    
    h_cache = {} if cnn_model is not None else None
    
    t_prep_maps_start = time.perf_counter()
    if cnn_model is not None:
        pad = patch_size // 2
        padded_grid = np.pad(grid, pad, mode='constant', constant_values=1)
        y_c, x_c = np.meshgrid(np.arange(size), np.arange(size), indexing='ij')
        max_dist = np.sqrt(2 * (patch_size**2))
        dist_map = np.sqrt((x_c - goal[1])**2 + (y_c - goal[0])**2)
        dist_map = np.clip(dist_map / max_dist, 0.0, 1.0)
        padded_dist = np.pad(dist_map, pad, mode='constant', constant_values=1.0)
    t_prep_maps_ms = (time.perf_counter() - t_prep_maps_start) * 1000
    
    n_expanded = 0
    n_cnn_calls = 0
    n_cache_hits = 0
    t_prepare_ms = t_prep_maps_ms
    t_infer_ms = 0.0
    t_queue_ms = 0.0
    
    t_total_start = time.perf_counter()
    
    while open_set:
        t_q_start = time.perf_counter()
        f, g, _, current = heapq.heappop(open_set)
        t_queue_ms += (time.perf_counter() - t_q_start) * 1000
        
        if current == goal:
            break
        if g > g_score.get(current, float('inf')):
            continue
            
        n_expanded += 1
        
        for neighbor, cost in get_neighbors_8way_strict(grid, current):
            tentative_g = g_score[current] + cost
            if tentative_g < g_score.get(neighbor, float('inf')):
                g_score[neighbor] = tentative_g
                came_from[neighbor] = current
                
                if cnn_model is not None:
                    if neighbor in h_cache:
                        h_cnn = h_cache[neighbor]
                        n_cache_hits += 1
                    else:
                        t_p_start = time.perf_counter()
                        px, py = neighbor
                        patch_obs = padded_grid[px : px+patch_size, py : py+patch_size]
                        patch_dist = padded_dist[px : px+patch_size, py : py+patch_size]
                        tensor_in = torch.tensor(np.stack([patch_obs, patch_dist], axis=0), dtype=torch.float32).unsqueeze(0)
                        t_prepare_ms += (time.perf_counter() - t_p_start) * 1000
                        
                        t_i_start = time.perf_counter()
                        with torch.no_grad():
                            h_cnn = cnn_model(tensor_in).item()
                        t_infer_ms += (time.perf_counter() - t_i_start) * 1000
                        n_cnn_calls += 1
                        h_cache[neighbor] = h_cnn
                    
                    h_eucl = np.sqrt((neighbor[0] - goal[0])**2 + (neighbor[1] - goal[1])**2)
                    h_used = max(h_cnn, h_eucl)
                else:
                    h_used = np.sqrt((neighbor[0] - goal[0])**2 + (neighbor[1] - goal[1])**2)
                
                f_score = tentative_g + h_used
                t_q_start = time.perf_counter()
                heapq.heappush(open_set, (f_score, tentative_g, counter, neighbor))
                t_queue_ms += (time.perf_counter() - t_q_start) * 1000
                counter += 1
                
    t_path_start = time.perf_counter()
    if current == goal:
        path = [current]
        while current in came_from:
            current = came_from[current]
            path.append(current)
        path = list(reversed(path))
        path_cost = g_score[goal]
        success = True
    else:
        path = []
        path_cost = -1
        success = False
    t_path_ms = (time.perf_counter() - t_path_start) * 1000
    
    stats = {
        "t_total_ms": (time.perf_counter() - t_total_start) * 1000,
        "t_prepare_ms": t_prepare_ms,
        "t_infer_ms": t_infer_ms,
        "t_queue_ms": t_queue_ms,
        "t_path_ms": t_path_ms,
        "n_expanded": n_expanded,
        "n_cnn_calls": n_cnn_calls,
        "n_cache_hits": n_cache_hits,
        "path_cost": path_cost,
        "success": success
    }
    return stats

# 5. ГЛАВНЫЙ ЦИКЛ ЭКСПЕРИМЕНТА
def run_experiment():
    print("=== ЗАПУСК ИСПРАВЛЕННОГО ЭКСПЕРИМЕНТА ===")
    print(json.dumps(get_env_info(), indent=2))
    
    os.makedirs("output_data", exist_ok=True)
    rng = np.random.default_rng(CONFIG["seed"])
    start, goal = (0,0), (CONFIG["grid_size"]-1, CONFIG["grid_size"]-1)
    
    all_train_maps, all_train_h_stars = [], []
    test_pools = {d: [] for d in CONFIG["densities"]}
    
    # 5.1 Генерация
    print("\n[1/4] Генерация карт...")
    for d in CONFIG["densities"]:
        count = 0
        attempts = 0
        while count < CONFIG["maps_per_density"] and attempts < CONFIG["maps_per_density"] * 20:
            grid = generate_map(CONFIG["grid_size"], d, rng)
            if is_reachable_strict(grid, start, goal):
                h_star_map = compute_dijkstra_heuristic_map(grid, goal)
                
                if h_star_map[start] != np.inf:
                    if rng.random() < 0.8:
                        all_train_maps.append(grid)
                        all_train_h_stars.append(h_star_map) 
                    else:
                        cost_opt = h_star_map[start]
                        test_pools[d].append((grid, cost_opt))
                    count += 1
            attempts += 1
        print(f"  Плотность {int(d*100)}%: {len(test_pools[d])} тестовых карт")

    
    print("\n[2/4] Обучение CNN...")
    dataset = HeuristicDataset(all_train_maps, all_train_h_stars, CONFIG["patch_size"])
    dataloader = DataLoader(dataset, batch_size=256, shuffle=True)
    cnn = LightweightCNN().to(CONFIG["device"])
    optimizer = optim.Adam(cnn.parameters(), lr=1e-3)
    criterion = nn.MSELoss()
    
    cnn.train()
    for epoch in range(CONFIG["cnn_epochs"]):
        total_loss = 0
        for inputs, targets in dataloader:
            inputs, targets = inputs.to(CONFIG["device"]), targets.to(CONFIG["device"])
            optimizer.zero_grad()
            loss = criterion(cnn(inputs).squeeze(), targets)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"  Epoch {epoch+1}/{CONFIG['cnn_epochs']}, Loss: {total_loss/len(dataloader):.4f}")
    
    cnn.eval() 

    print("\n[3/4] Прогрев CNN (warmup)...")
    dummy_input = torch.randn(1, 2, CONFIG["patch_size"], CONFIG["patch_size"], device=CONFIG["device"])
    for _ in range(CONFIG["warmup_runs"]):
        with torch.no_grad():
            _ = cnn(dummy_input)

    print("\n[4/4] Профилированное тестирование...")
    all_records = []
    
    for d in CONFIG["densities"]:
        print(f"\n Плотность {int(d*100)}% ")
        for map_idx, (grid, cost_opt) in enumerate(tqdm(test_pools[d], leave=False)):
            
            baseline_stats_list = []
            lh_stats_list = []
            
            for _ in range(CONFIG["measurement_runs"]):
                if random.random() < 0.5:
                    stats_b = astar_profiled(grid, start, goal, cnn_model=None)
                    stats_l = astar_profiled(grid, start, goal, cnn_model=cnn, patch_size=CONFIG["patch_size"])
                else:
                    stats_l = astar_profiled(grid, start, goal, cnn_model=cnn, patch_size=CONFIG["patch_size"])
                    stats_b = astar_profiled(grid, start, goal, cnn_model=None)
                
                baseline_stats_list.append(stats_b)
                lh_stats_list.append(stats_l)
            
            def med(key):
                return np.median([s[key] for s in lh_stats_list])
            def med_b(key):
                return np.median([s[key] for s in baseline_stats_list])
            
            all_records.append({
                "density": d,
                "map_id": map_idx,
                "cost_opt": cost_opt,
                "t_base_total_ms": med_b("t_total_ms"),
                "n_base_expanded": int(np.median([s["n_expanded"] for s in baseline_stats_list])),
                "t_lh_total_ms": med("t_total_ms"),
                "t_lh_prepare_ms": med("t_prepare_ms"),
                "t_lh_infer_ms": med("t_infer_ms"),
                "t_lh_queue_ms": med("t_queue_ms"),
                "n_lh_expanded": int(np.median([s["n_expanded"] for s in lh_stats_list])),
                "n_lh_cnn_calls": int(np.median([s["n_cnn_calls"] for s in lh_stats_list])),
                "n_lh_cache_hits": int(np.median([s["n_cache_hits"] for s in lh_stats_list])),
                "cost_lh": med("path_cost"),
                "success_lh": int(np.median([s["success"] for s in lh_stats_list]))
            })
            
    csv_path = "output_data/final_results.csv"
    with open(csv_path, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=all_records[0].keys())
        writer.writeheader()
        writer.writerows(all_records)
    print(f"\nРезультаты сохранены: {csv_path}")
    return all_records

# 6. АНАЛИЗ И ОБЪЯСНЕНИЕ МЕХАНИЗМА
def analyze_mechanism(records):
    print("ДЕКОМПОЗИЦИЯ ВРЕМЕНИ И МЕХАНИЗМ ЗАМЕДЛЕНИЯ")
    
    for d in CONFIG["densities"]:
        subset = [r for r in records if r["density"] == d]
        if not subset: continue
        
        n_maps = len(subset)
        t_base = np.mean([r["t_base_total_ms"] for r in subset])
        t_lh = np.mean([r["t_lh_total_ms"] for r in subset])
        t_lh_prep = np.mean([r["t_lh_prepare_ms"] for r in subset])
        t_lh_infer = np.mean([r["t_lh_infer_ms"] for r in subset])
        
        n_base_exp = np.mean([r["n_base_expanded"] for r in subset])
        n_lh_exp = np.mean([r["n_lh_expanded"] for r in subset])
        n_lh_cnn = np.mean([r["n_lh_cnn_calls"] for r in subset])
        n_lh_cache = np.mean([r["n_lh_cache_hits"] for r in subset])
        
        # Отношение длин путей к оптимуму
        ratios_l = [r["cost_lh"] / r["cost_opt"] for r in subset if r["cost_opt"] > 0]
        med_l_ratio = np.median(ratios_l)
        max_l_ratio = np.max(ratios_l) if ratios_l else 1.0
        n_suboptimal = sum(1 for r in ratios_l if r > 1.0001)
        n_violations = sum(1 for r in ratios_l if r > 1.05)
        
        print(f"\n--- Плотность {int(d*100)}% (N={n_maps}) ---")
        print(f"  Базовое A*:  T_total = {t_base:.2f} мс | N_expanded = {n_base_exp:.0f}")
        print(f"  A*-LH:       T_total = {t_lh:.2f} мс | N_expanded = {n_lh_exp:.0f}")
        print(f"    Декомпозиция A*-LH:")
        print(f"      T_prepare (тензоры) = {t_lh_prep:.2f} мс ({100*t_lh_prep/max(t_lh,1):.1f}%)")
        print(f"      T_infer (CNN)       = {t_lh_infer:.2f} мс ({100*t_lh_infer/max(t_lh,1):.1f}%)")
        print(f"    Счётчики CNN: Вызовов={n_lh_cnn:.0f}, Кэш-хитов={n_lh_cache:.0f}")
        print(f"    Качество пути: Медиана L_LH/L_Opt = {med_l_ratio:.4f} | Макс = {max_l_ratio:.4f}")
        print(f"    Неоптимальных путей: {n_suboptimal} из {n_maps} (Нарушений >5%: {n_violations})")
        
        if t_lh > t_base:
            overhead = t_lh - t_base
            print(f"   Замедление: +{overhead:.2f} мс")
            print(f"   Причина: T_prepare + T_infer = {t_lh_prep + t_lh_infer:.2f} мс ({100*(t_lh_prep+t_lh_infer)/overhead:.0f}% от overhead'а)")
            avg_call_time = (t_lh_prep + t_lh_infer) / max(n_lh_cnn, 1) * 1000
            print(f"   Среднее время одного вызова CNN: {avg_call_time:.1f} мкс")

if __name__ == "__main__":
    records = run_experiment()
    analyze_mechanism(records)
    print("\nЭКСПЕРИМЕНТ УСПЕШНО ЗАВЕРШЕН")

=== ЗАПУСК ФИНАЛЬНОГО ЭКСПЕРИМЕНТА ===
{
  "os": "Darwin 27.2.0",
  "processor": "arm",
  "python_version": "3.13.2",
  "torch_version": "2.14.1",
  "numpy_version": "2.5.4",
  "cpu_threads": 1,
  "device": "cpu",
  "note": "Single-threaded CPU mode is a REPRODUCIBILITY CONDITION, not a simulation of automotive edge hardware."
}

[1/4] Генерация карт...
  Плотность 10%: 92 тестовых карт
  Плотность 20%: 86 тестовых карт
  Плотность 30%: 54 тестовых карт
  Плотность 40%: 3 тестовых карт
  Плотность 50%: 0 тестовых карт

[2/4] Обучение CNN на 1103 картах...
